### 모델 상세 정보
URL: https://huggingface.co/openai/whisper-large-v3

- 위스퍼(Whisper)는 트랜스포머 기반의 인코더-디코더 모델로, 시퀀스-투-시퀀스 모델 이라고도 합니다 . 위스퍼 모델에는 영어 전용 버전과 다국어 지원 버전 두 가지가 있습니다. 영어 전용 모델은 영어 음성 인식 작업을 위해 학습되었고, 다국어 지원 모델은 다국어 음성 인식과 음성 번역 작업을 동시에 학습했습니다. 음성 인식의 경우, 모델은 오디오와 동일한 언어로 전사본을 예측합니다. 음성 번역의 경우, 모델은 오디오와 다른 언어 로 전사본을 예측합니다 .

- 위스퍼 체크포인트는 모델 크기가 다양한 5가지 구성으로 제공됩니다. 가장 작은 4개 구성은 영어 전용 및 다국어 버전으로 제공됩니다. 가장 큰 체크포인트는 다국어 전용입니다. 사전 학습된 10개의 체크포인트는 모두 허깅 페이스 허브(Hugging Face Hub) 에서 이용할 수 있습니다 . 체크포인트는 다음 표에 요약되어 있으며, 허브의 모델 링크도 함께 제공됩니다.



In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install --upgrade pip
!pip install --upgrade transformers datasets[audio] accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.7 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 96.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 18.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 67.6 MB/s  0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.1
    Uninstalling transformers-5.12.1:
      Successfully uninstalled transformers-5.12.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [tra

### 플래시 어텐션 2
- GPU가 Flash-Attention 2를 지원하고 torch.compile을 사용하지 않는 경우 Flash-Attention 2 사용을 권장합니다 .
- Flash-Attention 2를 사용하려면 먼저 Flash Attention을 설치하십시오 .

In [4]:
!pip install flash-attn --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 56.4 MB/s  0:00:00
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (pyproject.toml) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> flash-attn

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


- 그 다음 attn_implementation="flash_attention_2"으로 전달하세요 from_pretrained:

In [7]:
import torch
from transformers import AutoModelForSpeechSeq2Seq

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model_id = "openai/whisper-large-v3"

# Flash Attention 2 is a GPU-specific optimization and the package failed to install.
# Removing the attn_implementation argument to proceed without it.
model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True)

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

In [ ]:
# GPU가 Flash Attention을 지원하지 않는 경우, PyTorch의 SDPA(Scale Dot-Product Attention) 구현을
# 사용하는 것이 좋습니다.
# 이 어텐션 구현은 PyTorch 버전 2.1.1 이상에서 기본적으로 활성화되어 있습니다 .
# 호환되는 PyTorch 버전을 사용하고 있는지 확인하려면 다음 Python 코드 조각을 실행하세요.

# from transformers.utils import is_torch_sdpa_available

# print(is_torch_sdpa_available())

# model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, attn_implementation="sdpa")

In [3]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset


device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    dtype=torch_dtype,
    device=device,
    chunk_length_s=30,
)

dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation")
sample = dataset[0]["audio"]

# The audio sample is longer than 30 seconds, so return_timestamps must be set to True for long-form generation.
result = pipe(sample, return_timestamps=True)
print(result["text"])

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see re

KeyboardInterrupt: 

- 로컬 오디오 파일을 텍스트로 변환하려면 파이프라인을 호출할 때 오디오 파일의 경로를 전달하기만 하면 됩니다.

In [ ]:
result = pipe("/content/drive/MyDrive/AI/test.mp4")

- 여러 오디오 파일을 목록으로 지정하고 batch_size매개변수를 설정하면 여러 오디오 파일을 동시에 텍스트로 변환할 수 있습니다.

In [ ]:
result = pipe(["audio_1.mp3", "audio_2.mp3"], batch_size=2)

- Transformers는 온도 대체 및 이전 토큰 조건부 사용과 같은 모든 Whisper 디코딩 전략과 호환됩니다.
- 다음 예는 이러한 휴리스틱을 활성화하는 방법을 보여줍니다.

In [ ]:
generate_kwargs = {
    "max_new_tokens": 448,
    "num_beams": 1,
    "condition_on_prev_tokens": False,
    "compression_ratio_threshold": 1.35,  # zlib compression ratio threshold (in token space)
    "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
    "logprob_threshold": -1.0,
    "no_speech_threshold": 0.6,
    "return_timestamps": True,
}

result = pipe(sample, generate_kwargs=generate_kwargs)


- Whisper는 소스 오디오의 언어를 자동으로 예측합니다.
- 소스 오디오의 언어를 사전에 알고 있는 경우 파이프라인에 인수로 전달할 수 있습니다.

In [ ]:
result = pipe(sample, generate_kwargs={"language": "english"})

- 기본적으로 Whisper는 음성 소스 언어와 대상 텍스트 언어가 동일한 경우 음성 전사 작업을 수행합니다 .
- 대상 텍스트가 영어인 음성 번역을"translate" 수행하려면 작업을 다음과 같이 설정하십시오 .

In [ ]:
result = pipe(sample, generate_kwargs={"task": "translate"})

- 마지막으로, 이 모델을 사용하여 타임스탬프를 예측할 수 있습니다.
- 문장 수준의 타임스탬프를 얻으려면 return_timestamps다음 인수를 전달하세요.

In [ ]:
result = pipe(sample, return_timestamps=True)
print(result["chunks"])

- 단어 단위 타임스탬프는 다음과 같습니다.

In [ ]:
result = pipe(sample, return_timestamps="word")
print(result["chunks"])

- 위의 인수는 단독으로 또는 조합하여 사용할 수 있습니다.
- 예를 들어, 소스 오디오가 프랑스어이고 문장 수준의 타임스탬프를 반환하려는 음성 전사 작업을 수행하려면 다음을 사용할 수 있습니다.

In [ ]:
result = pipe(sample, return_timestamps=True, generate_kwargs={"language": "french", "task": "translate"})
print(result["chunks"])

### 생성 매개변수를 더욱 세밀하게 제어하려면 모델 + 프로세서 API를 직접 사용하십시오.

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
from datasets import Audio, load_dataset


device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

dataset = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
dataset = dataset.cast_column("audio", Audio(processor.feature_extractor.sampling_rate))
sample = dataset[0]["audio"]

inputs = processor(
    sample["array"],
    sampling_rate=sample["sampling_rate"],
    return_tensors="pt",
    truncation=False,
    padding="longest",
    return_attention_mask=True,
)
inputs = inputs.to(device, dtype=torch_dtype)

gen_kwargs = {
    "max_new_tokens": 448,
    "num_beams": 1,
    "condition_on_prev_tokens": False,
    "compression_ratio_threshold": 1.35,  # zlib compression ratio threshold (in token space)
    "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
    "logprob_threshold": -1.0,
    "no_speech_threshold": 0.6,
    "return_timestamps": True,
}

pred_ids = model.generate(**inputs, **gen_kwargs)
pred_text = processor.batch_decode(pred_ids, skip_special_tokens=True, decode_with_timestamps=False)

print(pred_text)


In [ ]:
# # 긴 형식의 덩어리
# Whisper의 수신 가능 시간은 30초입니다. 이보다 긴 오디오를 텍스트로 변환하려면 다음 두 가지 장형 알고리즘 중 하나가 필요합니다.

# 순차 방식: 버퍼링된 추론을 위해 "슬라이딩 윈도우"를 사용하며, 30초 단위로 순차적으로 데이터를 읽어들입니다.
# 청크 방식: 긴 오디오 파일을 더 짧은 파일들로 분할하고(세그먼트 간에 약간의 중복이 있음), 각 세그먼트를 독립적으로 전사한 후, 경계에서 결과 전사본을 합칩니다.
# 순차적 장문 알고리즘은 다음 두 가지 시나리오 중 하나에 사용해야 합니다.

# 전사 정확도가 가장 중요한 요소이며, 속도는 상대적으로 덜 중요한 고려 사항입니다.
# 긴 오디오 파일을 일괄 적 으로 전사하는 경우, 순차 전사 방식의 지연 시간은 청크 전사 방식과 비슷하지만, 단어 오류율(WER)은 최대 0.5% 더 높습니다.
# 반대로, 청크 알고리즘은 다음과 같은 경우에 사용해야 합니다.

# 전사 속도가 가장 중요한 요소입니다.
# 하나의 긴 오디오 파일을 텍스트로 변환하고 있습니다.
# 기본적으로 Transformers는 순차 알고리즘을 사용합니다. 청크 알고리즘을 활성화하려면 매개 chunk_length_s 변수를 전달하세요 pipeline. large-v3의 경우 청크 길이는 30초가 최적입니다. 긴 오디오 파일에 대한 배치 처리를 활성화하려면 인수를 전달하세요 batch_size.

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset


device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=30,
    batch_size=16,  # batch size for inference - set based on your device
    torch_dtype=torch_dtype,
    device=device,
)

dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation")
sample = dataset[0]["audio"]

result = pipe(sample)
print(result["text"])


- Gemini 수정본

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)

# --- 기존 데이터셋 불러오는 부분 삭제 ---
# dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation")
# sample = dataset[0]["audio"]

# --- 변경된 부분: 여기에 동영상(또는 오디오) 파일 경로를 입력하세요 ---
# Windows 예시: "C:/Users/사용자명/Videos/my_video.mp4" (백슬래시(\) 대신 슬래시(/) 사용 권장)
# Linux/Mac 예시: "/home/user/videos/my_video.mp4"
sample = "이곳에_동영상_파일_경로를_입력하세요.mp4"

# 파이프라인 실행
# 수정된 부분: chunk_length_s와 batch_size를 추가하여 메모리 초과 및 무한 대기를 방지합니다.
result = pipe(
    sample,
    return_timestamps=True,
    chunk_length_s=30,  # 30초 단위로 잘라서 처리
    batch_size=8,       # GPU 성능에 맞춰 8 또는 16으로 조정 (메모리 부족 시 4나 2로 낮춤)
)

# --- 새로 추가할 부분: 텍스트 파일로 저장 ---
# 추출된 텍스트를 변수에 담습니다.
extracted_text = result["text"]

# 화면에도 한 번 출력해서 확인합니다.
print(extracted_text)

# result.txt 라는 이름의 파일로 저장합니다.
# encoding="utf-8"을 설정해야 한글이 깨지지 않습니다.
with open("result.txt", "w", encoding="utf-8") as f:
    f.write(extracted_text)

print("✅ 'result.txt' 파일로 자막 저장이 완료되었습니다!")